In [ ]:
# 1) Centralize paths and data-loading setup at the top
import pandas as pd
import os
import glob
from pathlib import Path
import json

# Base directory for all related file paths
NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "gaze_ml_prediction.py").exists() and (NOTEBOOK_DIR / "UMAP").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "UMAP"
BASE_DIR = NOTEBOOK_DIR.parent
SOURCE_DIR = BASE_DIR / "user_data/user_data_final/responses"
RESPONSES_CSV = BASE_DIR / "responses.csv"
STIMULI_JSON = BASE_DIR / "user_data/stimuli_data.json"

print(f"Source folder path: {SOURCE_DIR}")
print(f"Output file path: {RESPONSES_CSV}")

# Check whether the source folder exists
if SOURCE_DIR.exists():
    print("✓ Source folder exists")
else:
    print("✗ Source folder does not exist")

# Find all CSV files
csv_pattern = str(SOURCE_DIR / "*.csv")
csv_files = glob.glob(csv_pattern)

print(f"Found {len(csv_files)} CSV files:")
for i, file in enumerate(csv_files, 1):
    filename = os.path.basename(file)
    print(f"{i:2d}. {filename}")

# Read all CSV files into a DataFrame list
dataframes = []
failed_files = []

for file in csv_files:
    try:
        df = pd.read_csv(
            file,
            dtype={
                'stimulus_id': 'string',
                'participant_id': 'string'
            },
            low_memory=False
        )
        dataframes.append(df)
        print(f"✓ Read successfully: {os.path.basename(file)} (rows: {len(df)}, columns: {len(df.columns)})")
    except Exception as e:
        failed_files.append((file, str(e)))
        print(f"✗ Failed to read: {os.path.basename(file)} - {str(e)}")

print(f"\nTotal: {len(dataframes)} files read successfully, {len(failed_files)} files failed to read")

if dataframes:
    # Merge all DataFrames
    combined_df = pd.concat(dataframes, ignore_index=True)
    print("Merge successful!")
    print(f"Merged data shape: {combined_df.shape}")
    print(f"Total rows: {len(combined_df)}")
    print(f"Total columns: {len(combined_df.columns)}")
    
    # Show column names
    print("\nColumn names:")
    for i, col in enumerate(combined_df.columns, 1):
        print(f"{i:2d}. {col}")
    
    # Preview the first few rows
    print("\nPreview of the first 5 rows:")
    print(combined_df.head())
else:
    print("No CSV files were read successfully; cannot merge.")

# Load stimuli_data.json and add the answer column to the response data
print("🔍 Loading stimuli_data.json...")
with open(STIMULI_JSON, 'r', encoding='utf-8') as f:
    stimuli_data = json.load(f)

# Build a qid -> answer mapping
qid_to_answer = {}
for item in stimuli_data:
    if isinstance(item, dict) and 'qid' in item and 'answer' in item:
        qid_to_answer[str(item['qid']).strip()] = item['answer']

print(f"✅ Found {len(qid_to_answer)} QID-Answer mappings")
print(f"   Sample mappings (first 3): {dict(list(qid_to_answer.items())[:3])}")

# Add the answer column to combined_df using the available key column
if 'combined_df' in locals() and combined_df is not None:
    candidate_key_cols = [c for c in ['stimulus_id', 'qid', 'question_id'] if c in combined_df.columns]
    key_col = candidate_key_cols[0] if candidate_key_cols else None

    print("\n📊 Adding answer column to response data...")
    print(f"   Original columns: {list(combined_df.columns)}")
    print(f"   Original shape: {combined_df.shape}")

    if key_col is None:
        print("❌ Matching key column not found (expected stimulus_id or qid or question_id)")
    else:
        # Normalize the key column as stripped strings
        combined_df[key_col] = combined_df[key_col].astype('string').str.strip()
        # Map answer values
        combined_df['answer'] = combined_df[key_col].map(qid_to_answer)
        
        # Check match quality
        matched_count = combined_df['answer'].notna().sum()
        total_count = len(combined_df)
        
        print(f"   ✅ Answer column added successfully via key '{key_col}'!")
        print(f"   ✅ Matched {matched_count}/{total_count} rows ({100*matched_count/total_count:.1f}%)")
        print(f"   ✅ New shape: {combined_df.shape}")
        print(f"   ✅ New columns: {list(combined_df.columns)}")

        # Show a few unmatched keys for debugging when needed
        if matched_count < total_count:
            unmatched_keys = combined_df.loc[combined_df['answer'].isna(), key_col].value_counts().head(5)
            print("\n⚠️ Unmatched sample keys (Top-5) and their counts:")
            print(unmatched_keys.to_dict())
        
        # Show a sample of the matched result
        print("\n📋 Sample data with answer column:")
        sample_cols = ['stimulus_id', 'answer', 'condition', 'reasoning_type'] if all(col in combined_df.columns for col in ['stimulus_id', 'answer', 'condition', 'reasoning_type']) else [key_col, 'answer']
        print(combined_df[sample_cols].head())
        
        # Show the answer distribution
        if matched_count > 0:
            print("\n📈 Answer distribution:")
            print(pd.Series(combined_df['answer']).map(lambda x: str(x)).value_counts())
else:
    print("❌ combined_df not found. Please run the previous cells first.")

# Save the merged data to the shared output path
if 'combined_df' in locals() and isinstance(combined_df, pd.DataFrame) and not combined_df.empty:
    try:
        combined_df.to_csv(RESPONSES_CSV, index=False, encoding='utf-8')
        print(f"✓ Successfully saved merged data to: {RESPONSES_CSV}")
        print(f"File size: {os.path.getsize(RESPONSES_CSV) / 1024 / 1024:.2f} MB")
        
        # Verify the saved file
        verification_df = pd.read_csv(RESPONSES_CSV, low_memory=False)
        print(f"Verification: the saved file contains {len(verification_df)} rows and {len(verification_df.columns)} columns")
    except Exception as e:
        print(f"✗ Error while saving file: {str(e)}")
else:
    print("No data available to save.")

### Statistical Preparation

Define experimental conditions and compute dependent measures.

In [ ]:
# 3) Statistical preparation using the shared response-data path
import numpy as np
import scipy.stats as stats
from scipy.stats import f_oneway
import seaborn as sns
import matplotlib.pyplot as plt

# Read the merged data from RESPONSES_CSV
df = pd.read_csv(
    RESPONSES_CSV,
    low_memory=False,
    dtype={
        'stimulus_id': 'string',
        'participant_id': 'string'
    },
)

print(f"Data shape: {df.shape}")
print(f"Number of participants: {df['participant_id'].nunique() if 'participant_id' in df.columns else 'Unknown'}")
print("\nData column names:")
print(df.columns.tolist())

# Define experimental conditions, including no_answer

def define_condition(row):
    cond = row.get('condition', None)
    trial_type = row.get('trial_type', None)
    reasoning_type = row.get('reasoning_type', None)

    # Baseline / pre trials without reasoning
    if cond == 'without_reasoning' and trial_type == 'pre':
        return 'no_answer'

    # Main trials without reasoning
    if cond == 'without_reasoning' and trial_type == 'main':
        return 'no_reasoning'

    # Reasoning trials
    if cond == 'with_reasoning' and reasoning_type == 'correct':
        return 'correct_reasoning'
    if cond == 'with_reasoning' and reasoning_type != 'correct':
        return 'incorrect_reasoning'  # reasoning_type != 'correct'

    return 'other'

# Create the condition column
df['experimental_condition'] = df.apply(define_condition, axis=1)
print("\nexperimental_condition counts:")
print(df['experimental_condition'].value_counts(dropna=False))

# Standardize decision/answer fields for downstream analysis
if 'decision' in df.columns:
    df['decision_norm'] = df['decision'].astype(str).str.strip().str.lower().map({'yes': True, 'no': False})
if 'answer' in df.columns:
    # In stimuli_data.json, answer is boolean; normalize if it was read back as a string
    df['answer_norm'] = df['answer'].map(lambda x: str(x).strip().lower()).map({'true': True, 'false': False})

# Compute dependent measures

# 1. Cognitive Load (2 items: cognitive_load_1, cognitive_load_2)
if set(['cognitive_load_1', 'cognitive_load_2']).issubset(df.columns):
    df['cognitive_load'] = df[['cognitive_load_1', 'cognitive_load_2']].apply(pd.to_numeric, errors='coerce').mean(axis=1)

# 2. Trust in Information (3 items: trust_1, trust_2, trust_3)
if set(['trust_1', 'trust_2', 'trust_3']).issubset(df.columns):
    df['trust_info'] = df[['trust_1', 'trust_2', 'trust_3']].apply(pd.to_numeric, errors='coerce').mean(axis=1)

# 3. Trust in System (6 items: trust_4 - trust_9)
trust_sys_cols = [c for c in ['trust_4', 'trust_5', 'trust_6', 'trust_7', 'trust_8', 'trust_9'] if c in df.columns]
if trust_sys_cols:
    df['trust_sys'] = df[trust_sys_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)

# 4. Reasoning Accuracy (2 items: reasoning_acc_1, reasoning_acc_2) — only meaningful in reasoning trials
reasoning_cols = [c for c in ['reasoning_acc_1', 'reasoning_acc_2'] if c in df.columns]
df['reasoning_acc'] = np.nan  # Initialize as NaN
reasoning_mask = df['experimental_condition'].isin(['correct_reasoning', 'incorrect_reasoning'])
if reasoning_cols:
    df.loc[reasoning_mask, 'reasoning_acc'] = df.loc[reasoning_mask, reasoning_cols].apply(pd.to_numeric, errors='coerce').mean(axis=1)

print("DV measures computation completed!")
if 'cognitive_load' in df.columns:
    print(f"Cognitive Load range: {df['cognitive_load'].min():.2f} - {df['cognitive_load'].max():.2f}")
if 'trust_info' in df.columns:
    print(f"Trust Info range: {df['trust_info'].min():.2f} - {df['trust_info'].max():.2f}")
if 'trust_sys' in df.columns:
    print(f"Trust System range: {df['trust_sys'].min():.2f} - {df['trust_sys'].max():.2f}")
if 'reasoning_acc' in df.columns:
    non_na_acc = df['reasoning_acc'].dropna()
    if len(non_na_acc) > 0:
        print(f"Reasoning Accuracy range: {non_na_acc.min():.2f} - {non_na_acc.max():.2f} (excluding NaN)")
    else:
        print("Reasoning Accuracy: all values are NaN (the data column may be missing or conditions may not match)")

### Descriptive Statistics


In [ ]:
# Compute descriptive statistics for each condition
# Existing DVs: cognitive_load, trust_info, trust_sys, reasoning_acc
# Added DVs: decision (match rate), confidence

# Ensure standardized decision/answer columns exist
if 'decision_norm' not in df.columns and 'decision' in df.columns:
    df['decision_norm'] = df['decision'].astype(str).str.strip().str.lower().map({'yes': True, 'no': False})
if 'answer_norm' not in df.columns and 'answer' in df.columns:
    df['answer_norm'] = df['answer'].map(lambda x: str(x).strip().lower()).map({'true': True, 'false': False})

# Safe helper for computing match rate
def decision_match_series(subset: pd.DataFrame):
    if 'decision_norm' not in subset.columns or 'answer_norm' not in subset.columns:
        return pd.Series(dtype='float64')
    valid = subset[['decision_norm', 'answer_norm']].dropna()
    if len(valid) == 0:
        return pd.Series(dtype='float64')
    return (valid['decision_norm'] == valid['answer_norm']).astype(int)


dv_measures = ['cognitive_load', 'trust_info', 'trust_sys', 'reasoning_acc', 'decision', 'confidence']
conditions = ['no_answer', 'no_reasoning', 'correct_reasoning', 'incorrect_reasoning']


def calculate_descriptives(data, condition, measure):
    """Compute descriptive statistics for one measure within one condition.
    reasoning_acc: not meaningful under no_reasoning/no_answer -> return N/A
    decision: compute the decision-vs-answer match rate
    others: standard numeric descriptives; return N/A if non-numeric
    """
    if measure == 'reasoning_acc' and condition in ['no_reasoning', 'no_answer']:
        return {'n': 'N/A', 'mean': 'N/A', 'std': 'N/A', 'min': 'N/A', 'max': 'N/A'}

    subset = data[data['experimental_condition'] == condition]

    if measure == 'decision':
        matches = decision_match_series(subset)
        if len(matches) == 0:
            return {'n': 0, 'mean': np.nan, 'std': np.nan, 'min': np.nan, 'max': np.nan}
        return {
            'n': len(matches),
            'mean': matches.mean(),   # Match rate
            'std': matches.std(ddof=1),
            'min': matches.min(),
            'max': matches.max()
        }

    # All other measures
    series = subset[measure] if measure in subset.columns else pd.Series(dtype='float64')
    if measure == 'reasoning_acc':
        series = pd.to_numeric(series, errors='coerce').dropna()
    numeric_series = pd.to_numeric(series, errors='coerce').dropna()
    if len(numeric_series) > 0:
        return {
            'n': len(numeric_series),
            'mean': numeric_series.mean(),
            'std': numeric_series.std(ddof=1),
            'min': numeric_series.min(),
            'max': numeric_series.max()
        }
    else:
        return {
            'n': len(series),
            'mean': 'N/A',
            'std': 'N/A',
            'min': 'N/A',
            'max': 'N/A'
        }

# Build the descriptive-statistics table
descriptive_stats = {}
for condition in conditions:
    descriptive_stats[condition] = {}
    for measure in dv_measures:
        descriptive_stats[condition][measure] = calculate_descriptives(df, condition, measure)

# Print the descriptive-statistics results
print("=" * 80)
print("Descriptive Statistics Results")
print("=" * 80)

for measure in dv_measures:
    print(f"\n【{measure.upper()}】")
    print("-" * 60)
    print(f"{'Condition':<20} {'N':<8} {'Mean':<10} {'Std':<10} {'Min':<8} {'Max':<8}")
    print("-" * 60)
    for condition in conditions:
        stats_row = descriptive_stats[condition][measure]
        if stats_row['n'] == 'N/A':
            print(f"{condition:<20} {'N/A':<8} {'N/A':<10} {'N/A':<10} {'N/A':<8} {'N/A':<8}")
        else:
            if isinstance(stats_row['mean'], (int, float, np.floating)) and not pd.isna(stats_row['mean']):
                mean_val = stats_row['mean'] if isinstance(stats_row['mean'], (int, float, np.floating)) else np.nan
                std_val = stats_row['std'] if isinstance(stats_row['std'], (int, float, np.floating)) else np.nan
                min_val = stats_row['min'] if isinstance(stats_row['min'], (int, float, np.floating)) else np.nan
                max_val = stats_row['max'] if isinstance(stats_row['max'], (int, float, np.floating)) else np.nan
                print(f"{condition:<20} {stats_row['n']:<8} {mean_val:<10.3f} {std_val:<10.3f} {min_val:<8.3f} {max_val:<8.3f}")
            else:
                print(f"{condition:<20} {stats_row['n']:<8} {'N/A':<10} {'N/A':<10} {'N/A':<8} {'N/A':<8}")
    print("-" * 60)

# Keep the more detailed cognitive-load statistics and normality checks below unchanged

### ANOVA and Post-hoc Tests (paired t-tests).

In [ ]:
# Participant-level repeated-measures ANOVA (prefer GG-corrected p), with paired post-hoc t-tests for all variables (reporting raw and FDR values)
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel
import pingouin as pg

# ---------- Helpers (use existing if defined; otherwise define) ----------
try:
    get_pid_col
    get_cond_col
except NameError:
    _PID_CANDIDATES = ['participant_id', 'participant', 'pid', 'subject_id', 'subject', 'P_ID', 'PId', 'PID']
    _COND_CANDIDATES = ['experimental_condition', 'condition', 'cond']
    def _get_col(df: pd.DataFrame, candidates: list[str], kind: str) -> str:
        for c in candidates:
            if c in df.columns:
                return c
        raise KeyError(f"Could not find {kind} column among candidates: {candidates}")
    def get_pid_col(df: pd.DataFrame) -> str:
        return _get_col(df, _PID_CANDIDATES, 'participant id')
    def get_cond_col(df: pd.DataFrame) -> str:
        return _get_col(df, _COND_CANDIDATES, 'condition')

pid_col = get_pid_col(df)
cond_col = get_cond_col(df)

# Ensure key derived columns exist/are numeric
# decision_match: prefer normalized if present; else derive from raw decision/answer when possible
if 'decision_match' not in df.columns:
    if {'decision_norm', 'answer_norm'}.issubset(df.columns):
        df['decision_match'] = (df['decision_norm'] == df['answer_norm']).astype(float)
    elif {'decision', 'answer'}.issubset(df.columns):
        dec_map = df['decision'].astype(str).str.strip().str.lower().map({'yes': True, 'no': False})
        ans_map = df['answer'].astype(str).str.strip().str.lower().map({'true': True, 'false': False})
        df['decision_match'] = (dec_map == ans_map).astype(float)
    else:
        df['decision_match'] = np.nan

# confidence numeric
if 'confidence' in df.columns:
    df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce')

# Measures to analyze (use decision_match instead of generic 'decision')
MEASURES_DEF = [
    'cognitive_load', 'trust_info', 'trust_sys', 'reasoning_acc', 'decision_match', 'confidence'
 ]

# Benjamini-Hochberg FDR correction (per-measure post-hoc family)
def bh_fdr(p_vals: list[float]) -> list[float]:
    p = np.array(p_vals, dtype=float)
    mask = ~np.isnan(p)
    m = mask.sum()
    if m == 0:
        return [np.nan] * len(p_vals)
    order = np.argsort(p[mask])
    ranked = p[mask][order]
    adj = np.empty_like(ranked)
    # work from the end to enforce monotonicity
    min_val = 1.0
    for i in range(m - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * m / rank
        min_val = min(min_val, val)
        adj[i] = min_val
    # place back into original positions
    p_adj = np.full_like(p, np.nan)
    p_adj_indices = np.where(mask)[0][order]
    p_adj[p_adj_indices] = np.minimum(adj, 1.0)
    return p_adj.tolist()

# Utility: run rmANOVA and return multiple p-variants for transparency
def rm_anova_participant_level(df_sub: pd.DataFrame, value_col: str, cond_order: list[str]) -> dict:
    # Aggregate participant×condition means
    vals = pd.to_numeric(df_sub[value_col], errors='coerce')
    tmp = pd.DataFrame({
        'val': vals,
        cond_col: df_sub[cond_col],
        pid_col: df_sub[pid_col],
    }).dropna(subset=['val', cond_col, pid_col])
    if tmp.empty:
        return {'F': np.nan, 'df1': np.nan, 'df2': np.nan, 'p_used': np.nan, 'p_used_label': None,
                'p_unc': np.nan, 'p_GG': np.nan, 'p_HF': np.nan, 'p_spher': np.nan,
                'n': 0, 'wide': pd.DataFrame()}

    # Keep only specified conditions and order
    tmp = tmp[tmp[cond_col].isin(cond_order)]
    if tmp.empty:
        return {'F': np.nan, 'df1': np.nan, 'df2': np.nan, 'p_used': np.nan, 'p_used_label': None,
                'p_unc': np.nan, 'p_GG': np.nan, 'p_HF': np.nan, 'p_spher': np.nan,
                'n': 0, 'wide': pd.DataFrame()}

    agg = tmp.groupby([pid_col, cond_col], as_index=False)['val'].mean()
    wide = agg.pivot(index=pid_col, columns=cond_col, values='val')
    # Require complete cases across the selected order
    wide = wide.reindex(columns=cond_order)
    wide = wide.dropna(how='any')
    if wide.shape[0] < 2:
        return {'F': np.nan, 'df1': np.nan, 'df2': np.nan, 'p_used': np.nan, 'p_used_label': None,
                'p_unc': np.nan, 'p_GG': np.nan, 'p_HF': np.nan, 'p_spher': np.nan,
                'n': wide.shape[0], 'wide': wide}

    long = wide.reset_index().melt(id_vars=[pid_col], var_name='cond', value_name='val')
    long['cond'] = pd.Categorical(long['cond'], categories=cond_order, ordered=True)

    try:
        aov = pg.rm_anova(dv='val', within='cond', subject=pid_col, data=long, detailed=True)
        row = aov.iloc[0]
        F = float(row['F']) if 'F' in row else np.nan
        df1 = float(row.get('ddof1', row.get('DF1', row.get('DF', np.nan))))
        df2 = float(row.get('ddof2', row.get('DF2', np.nan)))
        p_unc  = float(row.get('p-unc', np.nan))
        p_GG   = float(row.get('p-GG-corr', row.get('p-gg-corr', row.get('p-gg', np.nan))))
        p_HF   = float(row.get('p-HF-corr', row.get('p-hf-corr', row.get('p-hf', np.nan))))
        p_spher= float(row.get('p-spher', np.nan))
        # choose p_used: prefer GG if available, else p_unc
        if pd.notna(p_GG):
            p_used = p_GG
            p_used_label = 'GG-corr'
        elif pd.notna(p_unc):
            p_used = p_unc
            p_used_label = 'uncorrected'
        else:
            # last resort fallback (rare)
            p_used = p_spher if pd.notna(p_spher) else np.nan
            p_used_label = 'sphericity' if pd.notna(p_spher) else None
        return {
            'F': F, 'df1': df1, 'df2': df2,
            'p_used': float(p_used) if pd.notna(p_used) else np.nan,
            'p_used_label': p_used_label,
            'p_unc': p_unc if pd.notna(p_unc) else np.nan,
            'p_GG': p_GG if pd.notna(p_GG) else np.nan,
            'p_HF': p_HF if pd.notna(p_HF) else np.nan,
            'p_spher': p_spher if pd.notna(p_spher) else np.nan,
            'n': wide.shape[0], 'wide': wide
        }
    except Exception:
        return {'F': np.nan, 'df1': np.nan, 'df2': np.nan, 'p_used': np.nan, 'p_used_label': None,
                'p_unc': np.nan, 'p_GG': np.nan, 'p_HF': np.nan, 'p_spher': np.nan,
                'n': wide.shape[0], 'wide': wide}

def _pick_order_for_measure(measure: str) -> list[str]:
    # Default: use four conditions when possible; if `no_answer` is absent, fall back to three conditions automatically
    present = set(df[cond_col].dropna().astype(str).unique().tolist())
    if measure == 'reasoning_acc':
        return ['correct_reasoning', 'incorrect_reasoning']
    # `trust_*` usually has no ratings in pre / `no_answer`; prefer three conditions
    if measure in ('trust_info', 'trust_sys'):
        return ['no_reasoning', 'correct_reasoning', 'incorrect_reasoning']
    # For other measures, prefer four conditions
    if 'no_answer' in present:
        return ['no_answer', 'no_reasoning', 'correct_reasoning', 'incorrect_reasoning']
    return ['no_reasoning', 'correct_reasoning', 'incorrect_reasoning']

anova_rows: list[dict] = []
posthoc_rows: list[dict] = []

for m in MEASURES_DEF:
    order = _pick_order_for_measure(m)
    df_sub = df[df[cond_col].isin(order)].copy()

    # Run rmANOVA on participant-level means
    stats_res = rm_anova_participant_level(df_sub, m, order)
    is_sig = pd.notna(stats_res['p_used']) and (stats_res['p_used'] < 0.05)
    anova_rows.append({
        'measure': m, 'N': stats_res['n'], 'F': stats_res['F'], 'df1': stats_res['df1'], 'df2': stats_res['df2'],
        'p_used': stats_res['p_used'], 'p_label': stats_res['p_used_label'],
        'p_unc': stats_res['p_unc'], 'p_GG': stats_res['p_GG'], 'p_HF': stats_res['p_HF'],
        'anova_sig': bool(is_sig) if pd.notna(stats_res['p_used']) else False
    })

    # Pairwise t-tests for ALL measures (run them whether ANOVA is significant or not; treat non-significant ANOVA cases as exploratory)
    if isinstance(stats_res.get('wide'), pd.DataFrame) and not stats_res['wide'].empty:
        wide = stats_res['wide']
        conds = [c for c in order if c in wide.columns]
        for i in range(len(conds)):
            for j in range(i + 1, len(conds)):
                a, b = conds[i], conds[j]
                pair = wide[[a, b]].dropna()
                if pair.shape[0] >= 2:
                    t, p_pair = ttest_rel(pair[a], pair[b], nan_policy='omit')
                    df_pair = pair.shape[0] - 1
                else:
                    t, p_pair, df_pair = np.nan, np.nan, np.nan
                posthoc_rows.append({
                    'measure': m,
                    'cond_a': a,
                    'cond_b': b,
                    't': float(t) if pd.notna(t) else np.nan,
                    'df': int(df_pair) if pd.notna(df_pair) else np.nan,
                    'p_raw': float(p_pair) if pd.notna(p_pair) else np.nan,
                    'anova_sig': bool(is_sig) if pd.notna(stats_res['p_used']) else False,
                    'p_used': float(stats_res['p_used']) if pd.notna(stats_res['p_used']) else np.nan,
                    'p_label': stats_res.get('p_used_label')
                })

anova_df = pd.DataFrame(anova_rows).sort_values('p_used', na_position='last').reset_index(drop=True)
print("[Full dataset] Participant-level repeated-measures ANOVA results (sorted by selected p):")
star = lambda x: ("***" if x < 0.001 else "**" if x < 0.01 else "*" if x < 0.05 else "") if pd.notna(x) else ""
anova_df['stars'] = anova_df['p_used'].apply(star)
anova_df = anova_df[['measure','N','F','df1','df2','p_used','p_label','p_unc','p_GG','p_HF','anova_sig','stars']]
print(anova_df)
print("Number significant (p_used<.05):", int((anova_df['p_used'] < 0.05).sum()))

posthoc_df = pd.DataFrame(posthoc_rows)
if not posthoc_df.empty:
    # FDR within each measure's family of comparisons
    posthoc_df['p_fdr'] = np.nan
    for m, grp in posthoc_df.groupby('measure'):
        adj = bh_fdr(grp['p_raw'].tolist())
        posthoc_df.loc[grp.index, 'p_fdr'] = adj
    posthoc_df['sig_raw'] = posthoc_df['p_raw'] < 0.05
    posthoc_df['sig_fdr'] = posthoc_df['p_fdr'] < 0.05
    posthoc_df = posthoc_df.sort_values(['measure','p_raw'], na_position='last').reset_index(drop=True)

    posthoc_sig_df = posthoc_df[posthoc_df['anova_sig'] == True].copy()
    posthoc_nonsig_df = posthoc_df[posthoc_df['anova_sig'] == False].copy()

    print("\n[Full dataset] Post-hoc paired t-tests (variables with significant ANOVA; raw and FDR-BH):")
    if posthoc_sig_df.empty:
        print("  (none)")
    else:
        print(posthoc_sig_df[['measure','cond_a','cond_b','t','df','p_raw','sig_raw','p_fdr','sig_fdr']].head(100))

    print("\n[Full dataset] Post-hoc paired t-tests (variables with non-significant ANOVA | exploratory; raw and FDR-BH):")
    if posthoc_nonsig_df.empty:
        print("  (none)")
    else:
        print(posthoc_nonsig_df[['measure','cond_a','cond_b','t','df','p_raw','sig_raw','p_fdr','sig_fdr']].head(100))
else:
    print("\n[Full dataset] No post-hoc tests to run (insufficient samples or unable to construct paired data).")

In [ ]:
# Participant-level repeated-measures ANOVA (prefer GG-corrected p), with paired post-hoc t-tests for all variables (reporting raw and FDR values)
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel
import pingouin as pg

# ---------- Helpers (use existing if defined; otherwise define) ----------
try:
    get_pid_col
    get_cond_col
except NameError:
    _PID_CANDIDATES = ['participant_id', 'participant', 'pid', 'subject_id', 'subject', 'P_ID', 'PId', 'PID']
    _COND_CANDIDATES = ['experimental_condition', 'condition', 'cond']
    def _get_col(df: pd.DataFrame, candidates: list[str], kind: str) -> str:
        for c in candidates:
            if c in df.columns:
                return c
        raise KeyError(f"Could not find {kind} column among candidates: {candidates}")
    def get_pid_col(df: pd.DataFrame) -> str:
        return _get_col(df, _PID_CANDIDATES, 'participant id')
    def get_cond_col(df: pd.DataFrame) -> str:
        return _get_col(df, _COND_CANDIDATES, 'condition')

pid_col = get_pid_col(df)
cond_col = get_cond_col(df)

# Ensure key derived columns exist/are numeric
# decision_match: prefer normalized if present; else derive from raw decision/answer when possible
if 'decision_match' not in df.columns:
    if {'decision_norm', 'answer_norm'}.issubset(df.columns):
        df['decision_match'] = (df['decision_norm'] == df['answer_norm']).astype(float)
    elif {'decision', 'answer'}.issubset(df.columns):
        dec_map = df['decision'].astype(str).str.strip().str.lower().map({'yes': True, 'no': False})
        ans_map = df['answer'].astype(str).str.strip().str.lower().map({'true': True, 'false': False})
        df['decision_match'] = (dec_map == ans_map).astype(float)
    else:
        df['decision_match'] = np.nan

# confidence numeric
if 'confidence' in df.columns:
    df['confidence'] = pd.to_numeric(df['confidence'], errors='coerce')

# Measures to analyze (use decision_match instead of generic 'decision')
MEASURES_DEF = [
    'cognitive_load', 'trust_info', 'trust_sys', 'reasoning_acc', 'decision_match', 'confidence'
 ]

# Benjamini-Hochberg FDR correction (per-measure post-hoc family)
def bh_fdr(p_vals: list[float]) -> list[float]:
    p = np.array(p_vals, dtype=float)
    mask = ~np.isnan(p)
    m = mask.sum()
    if m == 0:
        return [np.nan] * len(p_vals)
    order = np.argsort(p[mask])
    ranked = p[mask][order]
    adj = np.empty_like(ranked)
    # work from the end to enforce monotonicity
    min_val = 1.0
    for i in range(m - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * m / rank
        min_val = min(min_val, val)
        adj[i] = min_val
    # place back into original positions
    p_adj = np.full_like(p, np.nan)
    p_adj_indices = np.where(mask)[0][order]
    p_adj[p_adj_indices] = np.minimum(adj, 1.0)
    return p_adj.tolist()

# Utility: run rmANOVA and return multiple p-variants for transparency
def rm_anova_participant_level(df_sub: pd.DataFrame, value_col: str, cond_order: list[str]) -> dict:
    # Aggregate participant×condition means
    vals = pd.to_numeric(df_sub[value_col], errors='coerce')
    tmp = pd.DataFrame({
        'val': vals,
        cond_col: df_sub[cond_col],
        pid_col: df_sub[pid_col],
    }).dropna(subset=['val', cond_col, pid_col])
    if tmp.empty:
        return {'F': np.nan, 'df1': np.nan, 'df2': np.nan, 'p_used': np.nan, 'p_used_label': None,
                'p_unc': np.nan, 'p_GG': np.nan, 'p_HF': np.nan, 'p_spher': np.nan,
                'n': 0, 'wide': pd.DataFrame()}

    # Keep only specified conditions and order
    tmp = tmp[tmp[cond_col].isin(cond_order)]
    if tmp.empty:
        return {'F': np.nan, 'df1': np.nan, 'df2': np.nan, 'p_used': np.nan, 'p_used_label': None,
                'p_unc': np.nan, 'p_GG': np.nan, 'p_HF': np.nan, 'p_spher': np.nan,
                'n': 0, 'wide': pd.DataFrame()}

    agg = tmp.groupby([pid_col, cond_col], as_index=False)['val'].mean()
    wide = agg.pivot(index=pid_col, columns=cond_col, values='val')
    # Require complete cases across the selected order
    wide = wide.reindex(columns=cond_order)
    wide = wide.dropna(how='any')
    if wide.shape[0] < 2:
        return {'F': np.nan, 'df1': np.nan, 'df2': np.nan, 'p_used': np.nan, 'p_used_label': None,
                'p_unc': np.nan, 'p_GG': np.nan, 'p_HF': np.nan, 'p_spher': np.nan,
                'n': wide.shape[0], 'wide': wide}

    long = wide.reset_index().melt(id_vars=[pid_col], var_name='cond', value_name='val')
    long['cond'] = pd.Categorical(long['cond'], categories=cond_order, ordered=True)

    try:
        aov = pg.rm_anova(dv='val', within='cond', subject=pid_col, data=long, detailed=True)
        row = aov.iloc[0]
        F = float(row['F']) if 'F' in row else np.nan
        df1 = float(row.get('ddof1', row.get('DF1', row.get('DF', np.nan))))
        df2 = float(row.get('ddof2', row.get('DF2', np.nan)))
        p_unc  = float(row.get('p-unc', np.nan))
        p_GG   = float(row.get('p-GG-corr', row.get('p-gg-corr', row.get('p-gg', np.nan))))
        p_HF   = float(row.get('p-HF-corr', row.get('p-hf-corr', row.get('p-hf', np.nan))))
        p_spher= float(row.get('p-spher', np.nan))
        # choose p_used: prefer GG if available, else p_unc
        if pd.notna(p_GG):
            p_used = p_GG
            p_used_label = 'GG-corr'
        elif pd.notna(p_unc):
            p_used = p_unc
            p_used_label = 'uncorrected'
        else:
            # last resort fallback (rare)
            p_used = p_spher if pd.notna(p_spher) else np.nan
            p_used_label = 'sphericity' if pd.notna(p_spher) else None
        return {
            'F': F, 'df1': df1, 'df2': df2,
            'p_used': float(p_used) if pd.notna(p_used) else np.nan,
            'p_used_label': p_used_label,
            'p_unc': p_unc if pd.notna(p_unc) else np.nan,
            'p_GG': p_GG if pd.notna(p_GG) else np.nan,
            'p_HF': p_HF if pd.notna(p_HF) else np.nan,
            'p_spher': p_spher if pd.notna(p_spher) else np.nan,
            'n': wide.shape[0], 'wide': wide
        }
    except Exception:
        return {'F': np.nan, 'df1': np.nan, 'df2': np.nan, 'p_used': np.nan, 'p_used_label': None,
                'p_unc': np.nan, 'p_GG': np.nan, 'p_HF': np.nan, 'p_spher': np.nan,
                'n': wide.shape[0], 'wide': wide}

def _pick_order_for_measure(measure: str) -> list[str]:
    # Only modification: add `no_answer` on top of the original setup (four conditions)
    # Note: `trust_info` / `trust_sys` usually have no scores in `no_answer` (pre); all-NaN values can reduce complete cases for four conditions to 0, causing ANOVA to return `NaN`
    if measure == 'reasoning_acc':
        return ['correct_reasoning', 'incorrect_reasoning']
    if measure in ('trust_info', 'trust_sys'):
        return ['no_reasoning', 'correct_reasoning', 'incorrect_reasoning']
    return ['no_answer', 'no_reasoning', 'correct_reasoning', 'incorrect_reasoning']

anova_rows: list[dict] = []
posthoc_rows: list[dict] = []

for m in MEASURES_DEF:
    order = _pick_order_for_measure(m)
    df_sub = df[df[cond_col].isin(order)].copy()

    # Run rmANOVA on participant-level means
    stats_res = rm_anova_participant_level(df_sub, m, order)
    is_sig = pd.notna(stats_res['p_used']) and (stats_res['p_used'] < 0.05)
    anova_rows.append({
        'measure': m, 'N': stats_res['n'], 'F': stats_res['F'], 'df1': stats_res['df1'], 'df2': stats_res['df2'],
        'p_used': stats_res['p_used'], 'p_label': stats_res['p_used_label'],
        'p_unc': stats_res['p_unc'], 'p_GG': stats_res['p_GG'], 'p_HF': stats_res['p_HF'],
        'anova_sig': bool(is_sig) if pd.notna(stats_res['p_used']) else False
    })

    # Pairwise t-tests for ALL measures (run them whether ANOVA is significant or not; treat non-significant ANOVA cases as exploratory)
    if isinstance(stats_res.get('wide'), pd.DataFrame) and not stats_res['wide'].empty:
        wide = stats_res['wide']
        conds = [c for c in order if c in wide.columns]
        for i in range(len(conds)):
            for j in range(i + 1, len(conds)):
                a, b = conds[i], conds[j]
                pair = wide[[a, b]].dropna()
                if pair.shape[0] >= 2:
                    t, p_pair = ttest_rel(pair[a], pair[b], nan_policy='omit')
                    df_pair = pair.shape[0] - 1
                else:
                    t, p_pair, df_pair = np.nan, np.nan, np.nan
                posthoc_rows.append({
                    'measure': m,
                    'cond_a': a,
                    'cond_b': b,
                    't': float(t) if pd.notna(t) else np.nan,
                    'df': int(df_pair) if pd.notna(df_pair) else np.nan,
                    'p_raw': float(p_pair) if pd.notna(p_pair) else np.nan,
                    'anova_sig': bool(is_sig) if pd.notna(stats_res['p_used']) else False,
                    'p_used': float(stats_res['p_used']) if pd.notna(stats_res['p_used']) else np.nan,
                    'p_label': stats_res.get('p_used_label')
                })

anova_df = pd.DataFrame(anova_rows).sort_values('p_used', na_position='last').reset_index(drop=True)
print("[Full dataset] Participant-level repeated-measures ANOVA results (sorted by selected p):")
star = lambda x: ("***" if x < 0.001 else "**" if x < 0.01 else "*" if x < 0.05 else "") if pd.notna(x) else ""
anova_df['stars'] = anova_df['p_used'].apply(star)
anova_df = anova_df[['measure','N','F','df1','df2','p_used','p_label','p_unc','p_GG','p_HF','anova_sig','stars']]
print(anova_df)
print("Number significant (p_used<.05):", int((anova_df['p_used'] < 0.05).sum()))

posthoc_df = pd.DataFrame(posthoc_rows)
if not posthoc_df.empty:
    # FDR within each measure's family of comparisons
    posthoc_df['p_fdr'] = np.nan
    for m, grp in posthoc_df.groupby('measure'):
        adj = bh_fdr(grp['p_raw'].tolist())
        posthoc_df.loc[grp.index, 'p_fdr'] = adj
    posthoc_df['sig_raw'] = posthoc_df['p_raw'] < 0.05
    posthoc_df['sig_fdr'] = posthoc_df['p_fdr'] < 0.05
    posthoc_df = posthoc_df.sort_values(['measure','p_raw'], na_position='last').reset_index(drop=True)

    posthoc_sig_df = posthoc_df[posthoc_df['anova_sig'] == True].copy()
    posthoc_nonsig_df = posthoc_df[posthoc_df['anova_sig'] == False].copy()

    print("\n[Full dataset] Post-hoc paired t-tests (variables with significant ANOVA; raw and FDR-BH):")
    if posthoc_sig_df.empty:
        print("  (none)")
    else:
        print(posthoc_sig_df[['measure','cond_a','cond_b','t','df','p_raw','sig_raw','p_fdr','sig_fdr']].head(100))

    print("\n[Full dataset] Post-hoc paired t-tests (variables with non-significant ANOVA | exploratory; raw and FDR-BH):")
    if posthoc_nonsig_df.empty:
        print("  (none)")
    else:
        print(posthoc_nonsig_df[['measure','cond_a','cond_b','t','df','p_raw','sig_raw','p_fdr','sig_fdr']].head(100))
else:
    print("\n[Full dataset] No post-hoc tests to run (insufficient samples or unable to construct paired data).")

In [ ]:
# Participant level: run a paired t-test only for `correct_reasoning` vs `incorrect_reasoning` (participant means across the two conditions)

# Keep only the two conditions that include reasoning
cond_a = 'correct_reasoning'
cond_b = 'incorrect_reasoning'
sub = df[df['experimental_condition'].isin([cond_a, cond_b])].copy()

# Map `decision` to numeric values (`Yes`=1, `No`=0), and generate an answer-matching measure when possible (use `decision_norm` / `answer_norm` if already available)
if 'decision' in sub.columns:
    sub['decision_num'] = (
        sub['decision'].astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0})
    )

if {'decision_norm', 'answer_norm'}.issubset(sub.columns):
    sub['decision_match'] = (sub['decision_norm'] == sub['answer_norm']).astype(float)
else:
    # If normalized columns are missing, `match` cannot be computed reliably, so leave it empty
    if 'decision_match' not in sub.columns:
        sub['decision_match'] = np.nan

# Infer the participant-ID column name (prefer `participant_id`)
if 'participant_id' in sub.columns:
    pid_col = 'participant_id'
elif 'pid' in sub.columns:
    pid_col = 'pid'
elif 'worker_id' in sub.columns:
    pid_col = 'worker_id'
else:
    raise ValueError('Participant ID column not found (e.g. participant_id/pid/worker_id)')

measures = {
    'trust_info': 'trust_info',
    'trust_sys': 'trust_sys',
    'cognitive_load': 'cognitive_load',
    'confidence': 'confidence',
    'reasoning_acc': 'reasoning_acc',  # Only meaningful under reasoning conditions
    'decision_num': 'decision_num',
    'decision_match': 'decision_match',
}

rows = []
for m, col in measures.items():
    if col not in sub.columns:
        rows.append({
            'measure': m, 'N': 0, 'mean_'+cond_a: np.nan, 'mean_'+cond_b: np.nan,
            't': np.nan, 'df': np.nan, 'p': np.nan, 'dz': np.nan, 'significant': False
        })
        continue

    vals = pd.to_numeric(sub[col], errors='coerce')
    tmp = pd.DataFrame({
        pid_col: sub[pid_col],
        'cond': sub['experimental_condition'],
        'val': vals,
    }).dropna(subset=['val', pid_col, 'cond'])

    if tmp.empty:
        rows.append({
            'measure': m, 'N': 0, 'mean_'+cond_a: np.nan, 'mean_'+cond_b: np.nan,
            't': np.nan, 'df': np.nan, 'p': np.nan, 'dz': np.nan, 'significant': False
        })
        continue

    agg = tmp.groupby([pid_col, 'cond'], as_index=False)['val'].mean()
    wide = agg.pivot(index=pid_col, columns='cond', values='val')

    # Keep only participants whose values are non-NaN in both conditions (`reindex` prevents `KeyError` when one condition is entirely missing for a measure)
    pair = wide.reindex(columns=[cond_a, cond_b]).dropna()
    if pair.empty or len(pair) < 2:
        rows.append({
            'measure': m, 'N': len(pair), 'mean_'+cond_a: pair[cond_a].mean() if not pair.empty else np.nan,
            'mean_'+cond_b: pair[cond_b].mean() if not pair.empty else np.nan,
            't': np.nan, 'df': np.nan, 'p': np.nan, 'dz': np.nan, 'significant': False
        })
        continue

    # Paired t-test (within-participant)
    t_stat, p_val = stats.ttest_rel(pair[cond_a], pair[cond_b], nan_policy='omit')

    # Within-participant effect size: Cohen's dz
    diff = pair[cond_a] - pair[cond_b]
    dz = np.nan
    if diff.std(ddof=1) > 0:
        dz = diff.mean() / diff.std(ddof=1)

    rows.append({
        'measure': m,
        'N': len(pair),
        'mean_'+cond_a: pair[cond_a].mean(),
        'mean_'+cond_b: pair[cond_b].mean(),
        't': t_stat,
        'df': len(pair) - 1,
        'p': p_val,
        'dz': dz,
        'significant': bool(p_val is not None and p_val < 0.05),
    })

res_df = pd.DataFrame(rows)
res_df = res_df.sort_values('p', na_position='last')

print(f"[Participant-level paired t-test] {cond_a} vs {cond_b}")
cols_to_show = ['measure', 'N', f'mean_{cond_a}', f'mean_{cond_b}', 't', 'df', 'p', 'dz', 'significant']
print(res_df[cols_to_show])